In [0]:
# ============================================================
# RETAIL MEDALLION PIPELINE — BRONZE LAYER
# Description: Ingests raw retail orders CSV into a Delta table with metadata columns for lineage tracking.
# ============================================================

# Define source and destination
RAW_FILE_PATH = "/Volumes/workspace/default/retail_raw_data/orders.csv"
BRONZE_TABLE_NAME = "bronze_orders"

print("Config loaded.")
print(f"   Source : {RAW_FILE_PATH}")
print(f"   Target : {BRONZE_TABLE_NAME}")

Config loaded.
   Source : /Volumes/workspace/default/retail_raw_data/orders.csv
   Target : bronze_orders


In [0]:
# ------------------------------------------------------------
# STEP 1: Read raw CSV from Unity Catalog Volume
# ------------------------------------------------------------
# We read the file exactly as-is — no transformations in Bronze.
# Bronze is the "raw vault" — preserving source data as received.

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(RAW_FILE_PATH)

print(f"Raw data loaded. Total records: {df_raw.count()}")
print(f"   Columns: {len(df_raw.columns)}")
df_raw.printSchema()

Raw data loaded. Total records: 9994
   Columns: 16
root
 |-- Order Id: integer (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub Category: string (nullable = true)
 |-- Product Id: string (nullable = true)
 |-- cost price: integer (nullable = true)
 |-- List Price: integer (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount Percent: integer (nullable = true)



In [0]:
# ------------------------------------------------------------
# STEP 2: Clean column names
# ------------------------------------------------------------
# Delta Lake does not allow spaces or special characters in
# column names. We standardize to snake_case here.

def clean_column_names(df):
    new_columns = [
        c.strip().lower().replace(" ", "_").replace("-", "_")
        for c in df.columns
    ]
    for old, new in zip(df.columns, new_columns):
        df = df.withColumnRenamed(old, new)
    return df

df_cleaned = clean_column_names(df_raw)

print("Column names standardized to snake_case.")
print(f"   Columns: {df_cleaned.columns}")

Column names standardized to snake_case.
   Columns: ['order_id', 'order_date', 'ship_mode', 'segment', 'country', 'city', 'state', 'postal_code', 'region', 'category', 'sub_category', 'product_id', 'cost_price', 'list_price', 'quantity', 'discount_percent']


In [0]:
# ------------------------------------------------------------
# STEP 3: Add metadata columns for data lineage
# ------------------------------------------------------------
# Every Bronze record gets tagged with:
#   - ingestion_timestamp : when the record was loaded
#   - source_file         : which file it came from
#   - pipeline_name       : which pipeline processed it
# This helps trace data issues back to the source later.

from pyspark.sql.functions import current_timestamp, lit

df_bronze = df_cleaned \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", lit(RAW_FILE_PATH)) \
    .withColumn("pipeline_name", lit("retail_medallion_pipeline"))

print(" Metadata columns added.")

 Metadata columns added.


In [0]:
# ------------------------------------------------------------
# STEP 4: Write to Delta Lake as Bronze managed table
# ------------------------------------------------------------
# We use saveAsTable() so Unity Catalog manages the storage
# location automatically — no manual path needed.
# mode("overwrite") allows re-running the pipeline safely.

df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(BRONZE_TABLE_NAME)

print(f" Bronze Delta table created: {BRONZE_TABLE_NAME}")

 Bronze Delta table created: bronze_orders


In [0]:
# ------------------------------------------------------------
# STEP 5: Verify & Profile the Bronze table
# ------------------------------------------------------------
count = spark.sql("SELECT COUNT(*) as total FROM bronze_orders").collect()[0][0]
print(f"Bronze table registered. Total records: {count}")
# Quick data profile
print("=== DATA PROFILE ===")
spark.sql(f"""
    SELECT
        COUNT(*)                        AS total_records,
        COUNT(DISTINCT order_id)        AS unique_orders,
        MIN(order_date)                 AS earliest_date,
        MAX(order_date)                 AS latest_date
    FROM {BRONZE_TABLE_NAME}
""").show()

Bronze table registered. Total records: 9994
=== DATA PROFILE ===
+-------------+-------------+-------------+-----------+
|total_records|unique_orders|earliest_date|latest_date|
+-------------+-------------+-------------+-----------+
|         9994|         9994|   2022-01-01| 2023-12-31|
+-------------+-------------+-------------+-----------+

